# 03c — Case-Control Matching (Problem C)

Match NTSB incidents (cases) to BTS normal flights (controls).  
This creates the labeled dataset for training the pre-flight risk model.

**Design:** 1:20 case-control ratio  
**Output:** `data/processed/preflight_casecontrol.parquet`

In [ ]:
import os
from pathlib import Path

root = Path.cwd()
while not (root / "pyproject.toml").exists():
    root = root.parent
os.chdir(root)
print(f"Working directory: {root}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

NTSB_PATH = Path('data/raw/ntsb_accidents.parquet')
BTS_DIR   = Path('data/raw/bts_flights')
OUT_DIR   = Path('data/processed')
OUT_DIR.mkdir(parents=True, exist_ok=True)

CONTROL_RATIO = 20  # controls per case
SEED = 42

print(f'Case-control ratio: 1:{CONTROL_RATIO}')

## 1. Load NTSB Cases

In [ ]:
ntsb = pd.read_parquet(NTSB_PATH)
print(f'NTSB events: {len(ntsb):,}')

# Ensure we have date and airport
ntsb['event_date'] = pd.to_datetime(ntsb['event_date'], errors='coerce')

# Standardize airport codes — NTSB uses FAA LIDs, BTS uses IATA 3-letter
# Many small airports have FAA LIDs that differ from IATA
# For major airports, FAA LID == IATA code (e.g., ATL, ORD, LAX)
airport_col = next((c for c in ntsb.columns if 'arpt' in c.lower() or 'airport' in c.lower()), None)
if airport_col:
    ntsb['airport_code'] = ntsb[airport_col].str.strip().str.upper()
    has_airport = ntsb['airport_code'].notna() & (ntsb['airport_code'] != '')
    print(f'Events with airport code: {has_airport.sum():,} ({has_airport.mean()*100:.1f}%)')
    ntsb_matched = ntsb[has_airport].copy()
else:
    print('WARNING: No airport column found')
    ntsb_matched = ntsb.copy()
    
print(f'\nMatchable NTSB events: {len(ntsb_matched):,}')

## 2. Load BTS Flights

In [ ]:
# Load BTS yearly parquets
bts_files = sorted(BTS_DIR.glob('bts_20??.parquet'))
print(f'BTS files found: {[f.name for f in bts_files]}')

bts_frames = []
for f in bts_files:
    df = pd.read_parquet(f)
    print(f'  {f.name}: {len(df):,} flights')
    bts_frames.append(df)

bts = pd.concat(bts_frames, ignore_index=True)
bts['FL_DATE'] = pd.to_datetime(bts['FL_DATE'], errors='coerce')
print(f'\nTotal BTS flights: {len(bts):,}')

## 3. Match NTSB Cases to BTS Flights

We match on **date + origin airport**. A match means an NTSB incident
occurred at the same airport on the same day as a scheduled BTS flight.

In [ ]:
# Create match keys
ntsb_matched['match_date'] = ntsb_matched['event_date'].dt.date
bts['match_date'] = bts['FL_DATE'].dt.date

# Build set of (date, airport) from NTSB
ntsb_keys = set(
    zip(ntsb_matched['match_date'], ntsb_matched['airport_code'])
)
print(f'Unique NTSB (date, airport) keys: {len(ntsb_keys):,}')

# Flag BTS flights that match an NTSB incident day+airport
bts['is_incident_day'] = [
    (d, a) in ntsb_keys 
    for d, a in zip(bts['match_date'], bts['ORIGIN'])
]

incident_flights = bts[bts['is_incident_day']].copy()
normal_flights   = bts[~bts['is_incident_day']].copy()

print(f'BTS flights on NTSB incident days/airports: {len(incident_flights):,}')
print(f'Normal flights: {len(normal_flights):,}')

## 4. Construct Case-Control Dataset

In [ ]:
# CASES: Mark incident-day flights as positive
cases = incident_flights.copy()
cases['incident'] = 1
n_cases = len(cases)
print(f'Cases: {n_cases:,}')

# CONTROLS: Sample from normal flights, stratified by month
n_controls_needed = n_cases * CONTROL_RATIO
print(f'Controls needed: {n_controls_needed:,} (1:{CONTROL_RATIO} ratio)')

# Stratified sampling by year-month to preserve temporal distribution
normal_flights['year_month'] = normal_flights['FL_DATE'].dt.to_period('M')
cases['year_month'] = cases['FL_DATE'].dt.to_period('M')

# Sample proportionally
case_month_dist = cases['year_month'].value_counts(normalize=True)

control_samples = []
for ym, frac in case_month_dist.items():
    n_sample = int(n_controls_needed * frac) + 1
    pool = normal_flights[normal_flights['year_month'] == ym]
    n_actual = min(n_sample, len(pool))
    sampled = pool.sample(n=n_actual, random_state=SEED)
    control_samples.append(sampled)

controls = pd.concat(control_samples, ignore_index=True)
controls['incident'] = 0
print(f'Controls sampled: {len(controls):,}')

# Combine
dataset = pd.concat([cases, controls], ignore_index=True)
dataset = dataset.drop(columns=['is_incident_day', 'match_date', 'year_month'], errors='ignore')
dataset = dataset.sample(frac=1, random_state=SEED).reset_index(drop=True)  # shuffle

print(f'\nFinal dataset: {len(dataset):,} rows')
print(f'Incident rate: {dataset["incident"].mean()*100:.2f}%')

## 5. Validate

In [ ]:
print('Label distribution:')
print(dataset['incident'].value_counts())

print(f'\nTemporal range: {dataset["FL_DATE"].min()} to {dataset["FL_DATE"].max()}')
print(f'Unique origins: {dataset["ORIGIN"].nunique()}')
print(f'Unique carriers: {dataset["OP_UNIQUE_CARRIER"].nunique()}')

# Sanity: cases should span the full date range
case_dates = dataset[dataset['incident'] == 1]['FL_DATE']
print(f'\nCase date range: {case_dates.min()} to {case_dates.max()}')

In [ ]:
# Save
out_path = OUT_DIR / 'preflight_casecontrol.parquet'
dataset.to_parquet(out_path, index=False)
print(f'Saved {len(dataset):,} rows to {out_path}')
print(f'File size: {out_path.stat().st_size / 1e6:.1f} MB')